# Motor Imagery EEG Signal Classification: MiniRocket vs CNN-LSTM
### Reproducing Hwaidi & Ghanem (*NeuroImage 328 (2026) 121816*)

This notebook demonstrates the end-to-end pipeline reproducing the paper's key findings:
1. **Signal Preprocessing**: Butterworth 4–40 Hz bandpass, Common Average Reference (CAR), baseline correction.
2. **Sensorimotor ERD/ERS Dynamics**: Inspecting $\mu$ (8–14 Hz) and $\beta$ (14–30 Hz) rhythm modulation over motor electrodes ($C3, C4, Cz$).
3. **MiniRocket Feature Extraction**: 10,000 dilated convolutional kernels + Proportion of Positive Values (PPV) pooling + RidgeClassifierCV.
4. **Hybrid CNN-LSTM Deep Learning**: Multi-scale 1D convolutions + 2-layer stacked LSTM with Adam optimizer and $L_2$ weight decay.
5. **Benchmarking**: Accuracy, Macro F1, and Inference Latency (demonstrating MiniRocket's ~13x speedup).

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))

from src.data_loader import generate_synthetic_eeg, PHYSIONET_CLASSES, load_physionet_subject
from src.preprocessing import preprocess_eeg_dataset
from src.minirocket_pipeline import MiniRocketPipeline
from src.cnn_lstm_model import CNNLSTMPipeline

print("Environment initialized successfully.")

## 1. Load and Preprocess Motor Imagery EEG Signals

In [ ]:
# Load PhysioNet 64-channel EEG data (or realistic ERD/ERS generator fallback)
X_raw, y_raw, sfreq, ch_names = load_physionet_subject(subject_id=1, downsample_sfreq=128.0)
print(f"Raw EEG shape: {X_raw.shape} | Trials: {len(y_raw)} | Channels: {len(ch_names)} | Sampling Rate: {sfreq} Hz")

# Preprocess: 4-40Hz Bandpass, CAR, Baseline (-0.5s to 0.0s), Z-score standardization
X_proc, y_proc = preprocess_eeg_dataset(
    X_raw, y_raw, sfreq=sfreq,
    l_freq=4.0, h_freq=40.0,
    notch_freq=50.0,
    apply_car=True,
    apply_baseline=True,
    apply_standardize=True
)
print("Preprocessing completed successfully.")

## 2. Visualize Sensorimotor Rhythms (C3 vs C4 Channels)

In [ ]:
time_vec = np.linspace(-0.5, 4.0, X_proc.shape[-1])
c3_idx = ch_names.index('C3') if 'C3' in ch_names else 0
c4_idx = ch_names.index('C4') if 'C4' in ch_names else 1

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

trial_idx = 0
axes[0].plot(time_vec, X_proc[trial_idx, c3_idx, :], label='C3 (Left Motor Area)', color='#1f77b4', lw=1.2)
axes[0].axvline(0.0, color='red', linestyle='--', label='Cue Onset (0s)')
axes[0].axvspan(0.0, 3.5, color='#FDE68A', alpha=0.3, label='Motor Imagery Window')
axes[0].set_ylabel('Amplitude (Z-score)')
axes[0].set_title(f'Trial {trial_idx} - True Class: {PHYSIONET_CLASSES[y_proc[trial_idx]]} - C3 Channel')
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].plot(time_vec, X_proc[trial_idx, c4_idx, :], label='C4 (Right Motor Area)', color='#ff7f0e', lw=1.2)
axes[1].axvline(0.0, color='red', linestyle='--')
axes[1].axvspan(0.0, 3.5, color='#FDE68A', alpha=0.3)
axes[1].set_ylabel('Amplitude (Z-score)')
axes[1].set_xlabel('Time relative to cue (seconds)')
axes[1].set_title(f'Trial {trial_idx} - C4 Channel')
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

## 3. MiniRocket: Feature Extraction & Ridge Classifier

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_proc, y_proc, test_size=0.2, random_state=42, stratify=y_proc)

# Fit MiniRocket with 2000 kernels
mr = MiniRocketPipeline(num_kernels=2000, random_state=42)
mr.fit(X_train, y_train)

mr_acc = mr.score(X_test, y_test)
mr_lat = mr.benchmark_latency(X_test[0:1], n_runs=20)
print(f"MiniRocket Test Accuracy: {mr_acc * 100:.2f}%")
print(f"MiniRocket Mean Latency:  {mr_lat['latency_mean_ms']:.2f} ms per trial")

## 4. Hybrid CNN-LSTM Deep Learning Model

In [ ]:
cnn = CNNLSTMPipeline(in_channels=X_proc.shape[1], n_classes=4, lr=1e-5, weight_decay=0.01)
cnn.fit(X_train, y_train, epochs=10, batch_size=16, verbose=True)

cnn_acc = cnn.score(X_test, y_test)
cnn_lat = cnn.benchmark_latency(X_test[0:1], n_runs=20)
print(f"CNN-LSTM Test Accuracy: {cnn_acc * 100:.2f}%")
print(f"CNN-LSTM Mean Latency:  {cnn_lat['latency_mean_ms']:.2f} ms per trial")

## 5. Latency and Speedup Comparison

In [ ]:
speedup = cnn_lat['latency_mean_ms'] / max(1e-4, mr_lat['latency_mean_ms'])
print(f"MiniRocket is {speedup:.1f}x FASTER than CNN-LSTM for real-time inference!")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['MiniRocket', 'CNN-LSTM'], [mr_lat['latency_mean_ms'], cnn_lat['latency_mean_ms']], color=['#1f77b4', '#d62728'])
ax.set_ylabel('Inference Latency (ms)')
ax.set_title('Single-Trial Latency Comparison')
plt.tight_layout()
plt.show()